<a href="https://colab.research.google.com/github/DhimanTarafdar/pytorch-nn-module/blob/main/pytorch_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn

Create a model class

In [6]:
class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear=nn.Linear(num_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear(features)
    out=self.sigmoid(out)
    return out

create dummy dataset and call forward pass

In [7]:
features=torch.rand(10,5)
model=Model(features.shape[1])
model(features)

tensor([[0.5305],
        [0.5214],
        [0.4092],
        [0.4003],
        [0.4410],
        [0.4376],
        [0.3677],
        [0.4617],
        [0.4910],
        [0.4455]], grad_fn=<SigmoidBackward0>)

In [8]:
model.linear.weight

Parameter containing:
tensor([[-0.3945, -0.1962, -0.0132, -0.3510, -0.3050]], requires_grad=True)

In [9]:
model.linear.bias

Parameter containing:
tensor([0.4056], requires_grad=True)

In [10]:
!pip install torchinfo

In [15]:
from torchinfo import summary
summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

Now create a nn with hidden layer (18 weights and 4 bias)



In [30]:
#again create model2
class Model2(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear1=nn.Linear(num_features,3)
    self.relu=nn.ReLU()
    self.linear2=nn.Linear(3,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear1(features)
    out=self.relu(out)
    out=self.linear2(out)
    out=self.sigmoid(out)
    return out

In [38]:
features=torch.rand(10,5)
model2=Model2(features.shape[1])
model2(features)

tensor([[0.5934],
        [0.5578],
        [0.5751],
        [0.5850],
        [0.5852],
        [0.5767],
        [0.5617],
        [0.5680],
        [0.5886],
        [0.5626]], grad_fn=<SigmoidBackward0>)

In [39]:
model2.linear1.weight

Parameter containing:
tensor([[ 0.2790,  0.2626, -0.0138, -0.3968,  0.0409],
        [ 0.2545,  0.1784,  0.4026,  0.0501, -0.3871],
        [-0.2908, -0.2101, -0.1047, -0.1182, -0.3817]], requires_grad=True)

In [40]:
model2.linear2.weight

Parameter containing:
tensor([[ 0.1892,  0.2438, -0.0606]], requires_grad=True)

In [41]:
model2.linear1.bias

Parameter containing:
tensor([0.3640, 0.0159, 0.1406], requires_grad=True)

In [42]:
model2.linear2.bias

Parameter containing:
tensor([0.1669], requires_grad=True)

In [43]:
summary(model2,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

effitient code for forward pass using sequential container

In [54]:
#again create model 3
class Model3(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,3),
        nn.ReLU(),
        nn.Linear(3,1),
        nn.Sigmoid()
    )


  def forward(self,features):
    out=self.network(features)
    return out

In [46]:
features=torch.rand(10,5)
model3=Model3(features.shape[1])
model3(features)

tensor([[0.6011],
        [0.6030],
        [0.6098],
        [0.5673],
        [0.5870],
        [0.5702],
        [0.5965],
        [0.5843],
        [0.5511],
        [0.5850]], grad_fn=<SigmoidBackward0>)

In [48]:
model3.network[0].weight

Parameter containing:
tensor([[-0.3131,  0.2963,  0.2036,  0.2523, -0.3371],
        [-0.0061, -0.1519,  0.0973,  0.0360,  0.4122],
        [ 0.0985, -0.4280,  0.0909,  0.2200, -0.0033]], requires_grad=True)

In [53]:
model3.network[2].weight

Parameter containing:
tensor([[ 0.3898,  0.3894, -0.5662]], requires_grad=True)

create training pipeline using nn module

In [56]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [57]:

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [58]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [59]:
df.head()


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [60]:
X_train,X_test,y_train,y_test= train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2,random_state=42)

In [61]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [63]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [64]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [75]:
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()

In [76]:
class MySimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear=nn.Linear(num_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear(features)
    out=self.sigmoid(out)
    return out

  def loss_function(self,y_pred,y):

    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss

In [73]:
learning_rate = 0.1
epochs = 25

In [81]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])
# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.linear.weight -= learning_rate * model.linear.weight.grad
    model.linear.bias -= learning_rate * model.linear.bias.grad

  # zero gradients
  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.7375736832618713
Epoch: 2, Loss: 0.7255277633666992
Epoch: 3, Loss: 0.7178683280944824
Epoch: 4, Loss: 0.7126966714859009
Epoch: 5, Loss: 0.7089020609855652
Epoch: 6, Loss: 0.7058777809143066
Epoch: 7, Loss: 0.703304648399353
Epoch: 8, Loss: 0.7010170817375183
Epoch: 9, Loss: 0.6989284157752991
Epoch: 10, Loss: 0.6969919800758362
Epoch: 11, Loss: 0.6951808333396912
Epoch: 12, Loss: 0.6934784650802612
Epoch: 13, Loss: 0.6918734312057495
Epoch: 14, Loss: 0.6903573870658875
Epoch: 15, Loss: 0.6889234185218811
Epoch: 16, Loss: 0.6875660419464111
Epoch: 17, Loss: 0.6862801313400269
Epoch: 18, Loss: 0.6850611567497253
Epoch: 19, Loss: 0.6839051842689514
Epoch: 20, Loss: 0.6828084588050842
Epoch: 21, Loss: 0.6817677617073059
Epoch: 22, Loss: 0.6807795763015747
Epoch: 23, Loss: 0.6798412203788757
Epoch: 24, Loss: 0.678949773311615
Epoch: 25, Loss: 0.6781029105186462


Built in loss function

In [82]:
class MySimpleNN2(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear=nn.Linear(num_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear(features)
    out=self.sigmoid(out)
    return out



In [87]:
loss_function=nn.BCELoss()

In [88]:
# create model
model2 = MySimpleNN2(X_train_tensor.shape[1])
# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model2(X_train_tensor)

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1))

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model2.linear.weight -= learning_rate * model2.linear.weight.grad
    model2.linear.bias -= learning_rate * model2.linear.bias.grad

  # zero gradients
  model2.linear.weight.grad.zero_()
  model2.linear.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.5565643310546875
Epoch: 2, Loss: 0.4499187469482422
Epoch: 3, Loss: 0.38752809166908264
Epoch: 4, Loss: 0.34648847579956055
Epoch: 5, Loss: 0.3171696960926056
Epoch: 6, Loss: 0.2949642241001129
Epoch: 7, Loss: 0.2774103283882141
Epoch: 8, Loss: 0.2630767226219177
Epoch: 9, Loss: 0.25107479095458984
Epoch: 10, Loss: 0.24082328379154205
Epoch: 11, Loss: 0.23192529380321503
Epoch: 12, Loss: 0.22410017251968384
Epoch: 13, Loss: 0.21714332699775696
Epoch: 14, Loss: 0.21090181171894073
Epoch: 15, Loss: 0.20525872707366943
Epoch: 16, Loss: 0.20012295246124268
Epoch: 17, Loss: 0.19542217254638672
Epoch: 18, Loss: 0.19109821319580078
Epoch: 19, Loss: 0.1871034801006317
Epoch: 20, Loss: 0.18339870870113373
Epoch: 21, Loss: 0.179951012134552
Epoch: 22, Loss: 0.1767326146364212
Epoch: 23, Loss: 0.1737198680639267
Epoch: 24, Loss: 0.1708924025297165
Epoch: 25, Loss: 0.16823258996009827
